In [ ]:
import re
import pandas as pd
import numpy as np
import pathlib
from bcpandas import SqlCreds, to_sql
import glob
import datetime
import os

# import xlsxwriter
import getpass
from datetime import date
import numpy as np
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl import load_workbook
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sqlalchemy import inspect
from sqlalchemy.types import NVARCHAR
from sqlalchemy import MetaData
from sqlalchemy import Column, Table
import pyodbc
import uuid

# these two imports are below and are CIDA specific
# script import start imports a number of functions as well as the user and a uuid
# we capture the user for the end so that we can build out the change directory, we also outside whatever main try/except block we use
# set up a failure counter to be used to push the data to Asan as necessary
from ScriptImportStart import (
    get_information,
    columnreplaceforsqlload,
    skiprowstofirstfullnonna,
    path_leaf,
    create_csv_from_html,
    tkinter_directory,
    error_function,
    joinstringsofdatframe,
    readmostrecentexcel,
    convertcolumnswithdateinname,
    continue_from_sql_index,
    find_files_created_today,
    LoadFilesFromFolder,
    LoadFilesFromFolderSwitchable,
)

# script montioring end imports all our script monitoring
# it pushes to sql and asana as a first passs and csv and email as a backup
# from ScriptMonitoringEnd import scriptmonitoring

from ScriptImportStart import user, failure, _uuid

os.chdir(
    r"C:\Users"
    + "\\"
    + user
    + "\Boston University\Continuous Improvement & Data Analytics - Documents\Automation and Data"
)

In [ ]:
server = "CIDA-SQL-T-01"
database = "spendcube"
username = "cidadata"
from spendcube import config
password = config.require("CIDA_SQL_PASSWORD")
schema = "dbo"
# Creates connection to sql server and the meta engine that will allow us to accss the tables
connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};SERVER="
    + server
    + ";DATABASE="
    + database
    + ";UID="
    + username
    + ";PWD="
    + password
)
connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})
engine = create_engine(
    connection_url, connect_args={"Encrypt": "no"}, fast_executemany=True
)

### the below is for the bigger entire loads as that way we cna do it using the  bulk copy program utility (bcp) which is faster

comballpath = pathlib.Path(
    (
        r"C:\Users"
        + "\\"
        + user
        + r"\Boston University\Continuous Improvement & Data Analytics - Documents\Automation and Data\Python-BCP"
    )
)


creds = SqlCreds(server, database, username, password)

In [ ]:
### Extra files for disambiguationn

gl = pd.read_excel(r"SAP-GL_Export\GL_EXPORT.XLSX")

gl = columnreplaceforsqlload(gl)
gl = gl[gl["chart_of_accounts"] == "0010"].loc[:, ["g/l_account", "g/l_acct_long_text"]]

supplierdetailsxlsx = glob.glob(
    "Output-Supplier List-S&P Version\Suppliers SandP*.xlsx"
)
supplierdetails = pd.read_excel(
    max(supplierdetailsxlsx, key=os.path.getctime), dtype=str
)

smallsupplierdetails = supplierdetails[["Supplier ID", "Line of Business"]]

columnreplaceforsqlload(smallsupplierdetails)

column_names = [
    "supplier",
    "gl_name",
    "product_description",
    "product_category",
    "line_of_business",
    "supplier_id",
    "transaction_num",
    "transaction_date",
    "funding_source",
    "amount",
    "datasource",
    "spendsource",
    "index",
    "join_key_datasource",
]

In [ ]:
# most recent disbursement file
disbursementblank = LoadFilesFromFolderSwitchable(
    "SAP HANA-ZPU_CP05 - Disbursements/Export*.csv",
    filetype="csv",
    filestoload="recent",
)

In [ ]:
# all disbursement file
disbursementblank = LoadFilesFromFolderSwitchable(
    "SAP HANA-ZPU_CP05 - Disbursements/Export*.csv",
    filetype="csv",
    filestoload="all",
)

In [ ]:
disb_table = "disbursements"
schemaandtable = schema + "." + disb_table

columnreplaceforsqlload(disbursementblank)
disbursementblank = disbursementblank.replace(np.nan, None)
disbursementblank.reset_index(drop=True, inplace=True)
disbursementblank = convertcolumnswithdateinname(disbursementblank)
disbursementblank[[c for c in disbursementblank if (c.find("zpu") == -1)]] = (
    disbursementblank[[c for c in disbursementblank if (c.find("zpu") == -1)]].astype(
        float
    )
)

disbursementblank.columns = disbursementblank.columns.str.replace(
    "zpu_cp05_v01", "disbursement"
)
disbursementblank = (
    disbursementblank.reset_index()
    .assign(
        DataSource="SAP HANA-ZPU_CP05_V01",
        SpendSource="Disbursements",
        disbursement_document_description_=disbursementblank.disbursement_document_description_.str.replace(
            "[^A-Za-z0-9 ]+", "", regex=True
        ),
        productcat="",
    )
    .assign(
        join_key_datasource=(
            lambda disbursementblank_: disbursementblank_.index.astype(str)
            + disbursementblank_.DataSource
        ),
        funding_source=disbursementblank.disbursement_funds_center_,
    )
    .merge(
        smallsupplierdetails,
        how="left",
        left_on="disbursement_vendor_",
        right_on="supplier_id",
    )
)

disbursementsmall = disbursementblank[
    [
        "disbursement_text_column_for_attribute_0vendor_",
        "disbursement_text_column_for_attribute_0gl_account_",
        "disbursement_document_description_",
        "productcat",
        "line_of_business",
        "disbursement_vendor_",
        "disbursement_fi_document_number_",
        "disbursement_posting_date_in_the_document_",
        "funding_source",
        "amount",
        "DataSource",
        "SpendSource",
        "index",
        "join_key_datasource",
    ]
]

disbursementsmall.columns = column_names

print("loadedtosql")

In [ ]:
# for loading most recent file
try:
    disbursementblank = continue_from_sql_index(
        schemaandtable, disbursementblank, engine
    )
except:
    pass


disbursementblank.to_sql(
    disb_table, engine, schema=schema, if_exists="append", index=True, chunksize=1000
)

In [ ]:
# for replacing all data
to_sql(
    disbursementblank,
    disb_table,
    creds,
    if_exists="replace",
    bcp_path=r"C:\Program Files\Microsoft SQL Server\Client SDK\ODBC\170\Tools\Binn\bcp.exe",
    work_directory=comballpath,
)

In [ ]:
srmlist = glob.glob(
    "SAP HANA-ZFSI_CP01 - SRM\Export*.csv"
)  # * means all if need specific format then *.csv
srmall = pd.DataFrame()
testlist = []
for file in srmlist:
    interim = pd.read_csv(file, dtype=str)
    srmall = pd.concat([interim, srmall], axis=0, join="outer")
columnreplaceforsqlload(srmall)
srmall = srmall.replace(np.nan, None)

srmall.columns = srmall.columns.str.replace("zsfi_cp01_v01_", "")
srmall.reset_index(drop=True, inplace=True)
srmall = convertcolumnswithdateinname(srmall)
srmall[[c for c in srmall if (c.find("value") != -1 or c.find("quantity") != -1)]] = (
    srmall[
        [c for c in srmall if (c.find("value") != -1 or c.find("quantity") != -1)]
    ].astype(float)
)
orderinvocievalue = srmall.groupby("srm_number_of_purchase_order_")[
    ["net_invoiced_value_in_invoice_currency"]
].sum()
orderinvoicesover5k = orderinvocievalue[
    orderinvocievalue.net_invoiced_value_in_invoice_currency >= 5000
]
orderinvoicesover5k = orderinvoicesover5k.assign(ApprovedByBuyer="S&P Approved")
orderinvoicesover5k = orderinvoicesover5k.drop(
    ["net_invoiced_value_in_invoice_currency"], axis=1
)
srmall = srmall.merge(
    orderinvoicesover5k, how="left", on="srm_number_of_purchase_order_"
)
conditions = [
    (srmall.srm_number_of_purchase_order_.astype("str").str.startswith("85")),
    (srmall.srm_number_of_purchase_order_.astype("str").str.startswith("75")),
    (srmall.srm_number_of_purchase_order_.astype("str").str.startswith("86")),
    (srmall.srm_number_of_purchase_order_.astype("str").str.startswith("76")),
]
values = ["Material", "Facilities Material", "Limit", "Facilities Limit"]
srmall = srmall.assign(srm_ordertype=np.select(conditions, values, default=""))
srmall = (
    srmall.assign(
        SpendSource=(
            srmall["ApprovedByBuyer"].astype(str)
            + " "
            + srmall["text_column_for_attribute_0bbp_cat_id"].astype(str)
            + " "
            + srmall["srm_ordertype"].astype(str)
        )
        .str.replace("nan", "")
        .str.replace("None", ""),
        DataSource="SAP HANA-ZFSI_CP01-SRM",
        srm_product_description_=srmall.srm_product_description_.replace(
            "[^A-Za-z0-9 ]+", "", regex=True
        ).replace("  ", " ", regex=True),
    )
    .reset_index()
    .assign(
        join_key_datasource=(
            lambda srmall_: srmall_.index.astype(str) + " " + srmall_.DataSource
        ),
        funding_source=srmall.srm_funds_center_,
    )
)

#

srmsmall = srmall[
    [
        "srm_text_column_for_attribute_0vendor_",
        "srm_text_column_for_attribute_0gl_account_",
        "srm_product_description_",
        "text_column_for_attribute_0bbp_cat_id",
        "type_of_business",
        "srm_vendor_",
        "srm_invoice_number_",
        "srm_sap_invoice_creation_date_",
        "funding_source",
        "net_invoiced_value_in_invoice_currency",
        "DataSource",
        "SpendSource",
        "index",
        "join_key_datasource",
    ]
]

srmsmall.columns = column_names

In [ ]:
srmall.to_sql("srm", engine, schema, if_exists="replace", index=True, chunksize=1000)

In [ ]:
# srmall.to_sql("srm", engine, if_exists="replace", index=True, chunksize=1000)

to_sql(
    srmall,
    "srm",
    creds,
    if_exists="replace",
    bcp_path=r"C:\Program Files\Microsoft SQL Server\Client SDK\ODBC\170\Tools\Binn\bcp.exe",
    work_directory=comballpath,
)

In [ ]:
# most recent boa file

boadf = LoadFilesFromFolderSwitchable(
    "BOA_PCARD-SpendViz/*Pcard*.txt",
    filetype="csv",
    filestoload="recent",
)

In [ ]:
# all boa file

boadf = LoadFilesFromFolderSwitchable(
    "BOA_PCARD-SpendViz/*Pcard*.txt",
    filetype="csv",
    filestoload="all",
)

In [ ]:
boa_table = "boa_pcard"
schemaandtable = schema + "." + boa_table

boadf = columnreplaceforsqlload(boadf)
boadf = boadf.reset_index(drop=True)
boadf = convertcolumnswithdateinname(boadf)
boadf[
    [
        c
        for c in boadf
        if (
            c.find("amount") != -1
            or c.find("quantity") != -1
            or c.find("total") != -1
            or c.find("_tax") != -1
        )
    ]
] = boadf[
    [
        c
        for c in boadf
        if (
            c.find("amount") != -1
            or c.find("quantity") != -1
            or c.find("total") != -1
            or c.find("_tax") != -1
        )
    ]
].astype(
    float
)

boadf = (
    boadf[~pd.isna(boadf.item_description)]
    .drop(
        ["detail_description", "detail_amount", "quantity", "detail_type"],
        axis="columns",
    )
    .drop_duplicates()
    .merge(
        boadf[(~pd.isna(boadf.detail_description)) & (boadf.detail_type != "Tax")][
            [
                "txn_number",
                "detail_description",
                "detail_amount",
                "quantity",
                "detail_type",
            ]
        ].drop_duplicates(),
        how="left",
        on="txn_number",
    )
)

boadf2 = (
    boadf.groupby("txn_number")
    .agg({"txn_number": "count", "amount": "mean"})
    # here we assign and round the approximate average by assigning each line an equal
    # amount of the total
    .assign(
        approx_avg=lambda df_: np.round(
            (df_.amount.astype(float) / df_.txn_number.astype(float)), 2
        )
    )
    # drop and merge back
    .rename({"txn_number": "txn_count"}, axis="columns")
    .reset_index(drop=False)
    .drop(["txn_count", "amount"], axis="columns")
    .merge(boadf, how="right", on="txn_number")
    .assign(
        DataSource="BankOfAmerica-PcardSpendviz",
        SpendSource="pCard",
        item_description=boadf.item_description.str.replace(
            "[^A-Za-z0-9 ]+", "", regex=True
        ),
        detail_description=boadf.detail_description.str.replace(
            "[^A-Za-z0-9 ]+", "", regex=True
        ),
        lineofbusiness="",
    )
    .reset_index()
)
boadf2 = boadf2.assign(
    combodescript=lambda boadf_: boadf_.item_description.fillna("")
    + " "
    + boadf_.detail_description.fillna(""),
    join_key_datasource=lambda boadf_: boadf_.index.astype(str)
    + " "
    + boadf_.DataSource,
    funding_source=boadf.gl__cost_object,
)


boasmall = boadf2[
    [
        "comp_supp_name",
        "gl__gl_account_desc",
        "combodescript",
        "mcc_description",
        "lineofbusiness",
        "vendor_id",
        "txn_number",
        "post_date",
        "funding_source",
        "approx_avg",
        "DataSource",
        "SpendSource",
        "index",
        "join_key_datasource",
    ]
]

boasmall.columns = column_names

# boadf["DataSource"] = "BankOfAmerica-PcardSpendviz"
# boadf["SpendSource"] = "pCard"
# cnxn = engine.connect()
# boadf.to_sql("boa_pcard", engine, if_exists="replace", index=True, chunksize=1000)
# cnxn.close()

In [ ]:
# load single file
try:
    boadf2 = continue_from_sql_index(schemaandtable, boadf2, engine)
except:
    pass

boadf2.to_sql(
    boa_table, engine, schema=schema, if_exists="append", index=True, chunksize=1000
)

In [ ]:
# load all and replace

to_sql(
    boadf2,
    "boa_pcard",
    creds,
    if_exists="replace",
    bcp_path=r"C:\Program Files\Microsoft SQL Server\Client SDK\ODBC\170\Tools\Binn\bcp.exe",
    work_directory=comballpath,
)

In [ ]:
# most recent ariba file

invoice_location = "Ariba-InvoceDetailsWDescription\*InvoiceDetailsWDescription*.csv"
newestinvoicedetail = LoadFilesFromFolderSwitchable(invoice_location, "csv", "recent")
newestinvoicedetail = newestinvoicedetail.drop(
    newestinvoicedetail.iloc[:, 0].name, axis="columns"
)

In [ ]:
# all ariba files

invoice_location_csv = (
    "Ariba-InvoceDetailsWDescription\\*InvoiceDetailsWDescription*.csv"
)
newestinvoicedetail_csv = LoadFilesFromFolderSwitchable(
    invoice_location_csv, "csv", "all"
)

invoice_location_html = (
    "Ariba-InvoceDetailsWDescription\*InvoiceDetailsWDescription*.xls"
)
# newestinvoicedetail_html = LoadFilesFromFolderSwitchable(invoice_location_html,'html','all')

# newestinvoicedetail = pd.concat(
#    [newestinvoicedetail_html.reset_index(drop=True), newestinvoicedetail_csv.reset_index(drop=True)], axis=0
# ).drop_duplicates()

newestinvoicedetail = newestinvoicedetail_csv

In [ ]:
newestinvoicedetail = columnreplaceforsqlload(newestinvoicedetail)

newestinvoicedetail = convertcolumnswithdateinname(newestinvoicedetail)

newestinvoicedetail[[c for c in newestinvoicedetail if (c.startswith("sum"))]] = (
    newestinvoicedetail[[c for c in newestinvoicedetail if (c.startswith("sum"))]]
    .astype(str)
    .apply(lambda x: x.str.replace(",", ""))
    .apply(lambda x: x.str.replace("NaT", ""))
    .astype(float, errors="ignore")
)

newestinvoicedetail = newestinvoicedetail.reset_index(drop=True)

linetypeconditons = [
    (newestinvoicedetail.inv_line_type == "Punch-Out Item"),
    (newestinvoicedetail.inv_line_type == "Catalog Item"),
    (newestinvoicedetail.inv_line_type == "Non-Catalog Item"),
    (newestinvoicedetail.inv_line_type == "Fee Item"),
    (newestinvoicedetail.inv_line_type == "Misc Expense Item"),
    (newestinvoicedetail.inv_line_type == "Freight Charge"),
    (newestinvoicedetail.inv_line_type == "Handling Charge"),
    (newestinvoicedetail.inv_line_type == "Discount"),
    (newestinvoicedetail.inv_line_type == "Value Added Tax"),
    (newestinvoicedetail.inv_line_type == "Withholding Tax"),
    (newestinvoicedetail.inv_line_type == "SalesTax"),
]

linetypevalues = [
    "Catalog",
    "Catalog",
    "Non-Catalog",
    "BPO Scheduled Payment",
    "BPO Expense",
    "Charge",
    "Charge",
    "Discount",
    "Tax",
    "Tax",
    "Tax",
]

newestinvoicedetail["Updatedlinetype"] = np.select(
    linetypeconditons, linetypevalues, default=""
)

In [ ]:
newestinvoicedetail["Updatedlinetype"] = np.select(
    linetypeconditons, linetypevalues, default=""
)
buyerhtml = glob.glob("Ariba-Purchase Order Details\Purchase*.xls")
buyerdf = LoadFilesFromFolder(buyerhtml, "html")

buyercsv = glob.glob("Ariba-Purchase Order Details\Purchase*.csv")
buyerdf2 = LoadFilesFromFolder(buyercsv, "csv")

buyerdf = pd.concat([buyerdf, buyerdf2], axis=0, join="outer").copy()

buyerdf = columnreplaceforsqlload(buyerdf)

orderswithspend = buyerdf[["order_id", "sum_po_spend_"]]
orderswithspend["sum_po_spend_"] = (
    orderswithspend["sum_po_spend_"].astype(str).str.replace(",", "").astype(float)
)
orderswithspend = orderswithspend.groupby("order_id")[["sum_po_spend_"]].sum()
buyerapprovedorders = orderswithspend[orderswithspend["sum_po_spend_"] >= 10000]
buyerapprovedorders = buyerapprovedorders.assign(ApprovedByBuyer="S&P Approved")

orderstype = buyerdf[["order_id", "order_type_1"]].copy()
orderstype = orderstype.sort_values(by=["order_type_1"]).drop_duplicates(
    subset=["order_id"]
)
allorders = orderstype.merge(buyerapprovedorders, how="left", on="order_id")
allorders = allorders.assign(
    ordertypeandapproval=(
        allorders.order_type_1.astype(str) + " " + allorders.ApprovedByBuyer.astype(str)
    ).str.replace("nan", "")
)
allorders = allorders[allorders["order_id"] != "Unclassified"]
allorderssmall = allorders[["order_id", "ordertypeandapproval"]]
newestinvoicedetailordertypes = newestinvoicedetail.merge(
    allorderssmall, how="left", left_on="inv_order_id", right_on="order_id"
)
newestinvoicedetailordertypes = newestinvoicedetailordertypes.assign(
    SpendSource=(
        newestinvoicedetailordertypes.inv_invoice_source_document.astype(str)
        + " "
        + newestinvoicedetailordertypes.Updatedlinetype.astype(str)
        + " "
        + newestinvoicedetailordertypes.ordertypeandapproval.astype(str)
    )
    .str.replace("nan", "")
    .str.replace("None", "")
    .str.strip(),
    DataSource="Ariba-Invoice Details With Description",
)


In [ ]:
newestinvoicedetailordertypes[newestinvoicedetailordertypes.SpendSource == '']

In [ ]:
ariba_table = "ariba"
schemaandtable = schema + "." + ariba_table


def assign_values(row, column_name_1, column_name_2, column_name_3):
    if row[column_name_1] != "Unclassified":
        return row[column_name_1]
    elif row[column_name_2] != "Unclassified":
        return row[column_name_2]
    else:
        return row[column_name_3]


newestinvoicedetail = columnreplaceforsqlload(newestinvoicedetail)

newestinvoicedetail = convertcolumnswithdateinname(newestinvoicedetail)

newestinvoicedetail[[c for c in newestinvoicedetail if (c.startswith("sum"))]] = (
    newestinvoicedetail[[c for c in newestinvoicedetail if (c.startswith("sum"))]]
    .astype(str)
    .apply(lambda x: x.str.replace(",", ""))
    .apply(lambda x: x.str.replace("NaT", ""))
    .astype(float, errors="ignore")
)

newestinvoicedetail = newestinvoicedetail.reset_index(drop=True)

linetypeconditons = [
    (newestinvoicedetail.inv_line_type == "Punch-Out Item"),
    (newestinvoicedetail.inv_line_type == "Catalog Item"),
    (newestinvoicedetail.inv_line_type == "Non-Catalog Item"),
    (newestinvoicedetail.inv_line_type == "Fee Item"),
    (newestinvoicedetail.inv_line_type == "Misc Expense Item"),
    (newestinvoicedetail.inv_line_type == "Freight Charge"),
    (newestinvoicedetail.inv_line_type == "Handling Charge"),
    (newestinvoicedetail.inv_line_type == "Discount"),
    (newestinvoicedetail.inv_line_type == "Value Added Tax"),
    (newestinvoicedetail.inv_line_type == "Withholding Tax"),
    (newestinvoicedetail.inv_line_type == "SalesTax"),
]

linetypevalues = [
    "Catalog",
    "Catalog",
    "Non-Catalog",
    "BPO Scheduled Payment",
    "BPO Expense",
    "Charge",
    "Charge",
    "Discount",
    "Tax",
    "Tax",
    "Tax",
]

newestinvoicedetail["Updatedlinetype"] = np.select(
    linetypeconditons, linetypevalues, default=""
)
buyerhtml = glob.glob("Ariba-Purchase Order Details\Purchase*.xls")
buyerdf = LoadFilesFromFolder(buyerhtml, "html")

buyercsv = glob.glob("Ariba-Purchase Order Details\Purchase*.csv")
buyerdf2 = LoadFilesFromFolder(buyercsv, "csv")

buyerdf = pd.concat([buyerdf, buyerdf2], axis=0, join="outer").copy()

buyerdf = columnreplaceforsqlload(buyerdf)

orderswithspend = buyerdf[["order_id", "sum_po_spend_"]]
orderswithspend["sum_po_spend_"] = (
    orderswithspend["sum_po_spend_"].astype(str).str.replace(",", "").astype(float)
)
orderswithspend = orderswithspend.groupby("order_id")[["sum_po_spend_"]].sum()
buyerapprovedorders = orderswithspend[orderswithspend["sum_po_spend_"] >= 10000]
buyerapprovedorders = buyerapprovedorders.assign(ApprovedByBuyer="S&P Approved")

orderstype = buyerdf[["order_id", "order_type_1"]].copy()
orderstype = orderstype.sort_values(by=["order_type_1"]).drop_duplicates(
    subset=["order_id"]
)
allorders = orderstype.merge(buyerapprovedorders, how="left", on="order_id")
allorders = allorders.assign(
    ordertypeandapproval=(
        allorders.order_type_1.astype(str) + " " + allorders.ApprovedByBuyer.astype(str)
    ).str.replace("nan", "")
)
allorders = allorders[allorders["order_id"] != "Unclassified"]
allorderssmall = allorders[["order_id", "ordertypeandapproval"]]
newestinvoicedetailordertypes = newestinvoicedetail.merge(
    allorderssmall, how="left", left_on="inv_order_id", right_on="order_id"
)
newestinvoicedetailordertypes = newestinvoicedetailordertypes.assign(
    SpendSource=(
        newestinvoicedetailordertypes.inv_invoice_source_document.astype(str)
        + " "
        + newestinvoicedetailordertypes.Updatedlinetype.astype(str)
        + " "
        + newestinvoicedetailordertypes.ordertypeandapproval.astype(str)
    )
    .str.replace("nan", "")
    .str.replace("None", "")
    .str.strip(),
    DataSource="Ariba-Invoice Details With Description",
)

newestinvoicedetailordertypes = (
    newestinvoicedetailordertypes.assign(
        inv_description=newestinvoicedetailordertypes.inv_description.str.replace(
            "[^A-Za-z0-9 ]+", "", regex=True
        ),
        invsupplier__erp_supplier_id_=newestinvoicedetailordertypes.invsupplier__erp_supplier_id_.str.replace(
            "'", ""
        ),
    )
    .merge(
        smallsupplierdetails,
        how="left",
        left_on="invsupplier__erp_supplier_id_",
        right_on="supplier_id",
    )
    .reset_index()
    .assign(
        join_key_datasource=lambda newestinvoicedetailordertypes_: newestinvoicedetailordertypes_.index.astype(
            str
        )
        + " "
        + newestinvoicedetailordertypes_.DataSource
    )
)

newestinvoicedetailordertypes["funding_source"] = newestinvoicedetailordertypes.apply(
    lambda row: assign_values(
        row,
        "invinternal_order__internal_order_id_",
        "invwbselement__wbselement_id_",
        "invcost_center__cost_center_id_",
    ),
    axis=1,
)

try:
    newestinvoicedetailordertypes = newestinvoicedetailordertypes.drop(
        columns="updatedlinetype"
    )
except:
    pass

aribasmall = newestinvoicedetailordertypes[
    [
        "invsupplier__erp_supplier_",
        "invgl_account__gl_account_name_",
        "inv_description",
        "inverp_commodity__product_category_",
        "line_of_business",
        "invsupplier__erp_supplier_id_",
        "ipt_erp_invoice_number",
        "invaccounting_date__date_",
        "funding_source",
        "sum_invoice_spend_",
        "DataSource",
        "SpendSource",
        "index",
        "join_key_datasource",
    ]
]

aribasmall.columns = column_names

# newestinvoicedetailordertypes.to_sql(
#    "ariba", engine, if_exists="replace", index=True, chunksize=1000
# )

In [ ]:
# load only one file
try:
    newestinvoicedetailordertypes = continue_from_sql_index(
        schemaandtable, newestinvoicedetailordertypes, engine
    )
except:
    pass
newestinvoicedetailordertypes.to_sql(
    ariba_table, engine, schema=schema, if_exists="append", index=True, chunksize=1000
)
print(datetime.datetime.utcnow())

In [ ]:
# load all and replace

to_sql(
    newestinvoicedetailordertypes,
    "ariba",
    creds,
    if_exists="replace",
    bcp_path=r"C:\Program Files\Microsoft SQL Server\Client SDK\ODBC\170\Tools\Binn\bcp.exe",
    work_directory=comballpath,
)

In [ ]:
# most recent concur file

concur_entry_location = "Concur-Spendcube-Entries/*SpendCube-Entries*.csv"
concur_allocation_location = (
    "Concur-SpendcubeEntriesAllocations/*SpendCube-Entries-Allocation*.csv"
)
concurnewestfile = LoadFilesFromFolderSwitchable(
    regexfilepath=concur_entry_location,
    filetype="csv",
    filestoload="recent",
    file_encoding="utf_16",
)
concurallocationnewestfile = LoadFilesFromFolderSwitchable(
    regexfilepath=concur_allocation_location,
    filetype="csv",
    filestoload="recent",
    file_encoding="utf_16",
)

In [ ]:
# all concur files

concur_entry_location = "Concur-Spendcube-Entries/*SpendCube-Entries*.csv"
concur_allocation_location = (
    "Concur-SpendcubeEntriesAllocations/*SpendCube-Entries-Allocation*.csv"
)
concurnewestfile = LoadFilesFromFolderSwitchable(
    regexfilepath=concur_entry_location,
    filetype="csv",
    filestoload="all",
    file_encoding="utf_16",
)
concurallocationnewestfile = LoadFilesFromFolderSwitchable(
    regexfilepath=concur_allocation_location,
    filetype="csv",
    filestoload="all",
    file_encoding="utf_16",
)

In [ ]:
concur_table = "concur"

scheamandtable = schema + "." + concur_table

concurnewestfile = concurnewestfile[concurnewestfile["Transaction Type"] != "Parent"]

concurnewestfile = columnreplaceforsqlload(concurnewestfile)

concurnewestfile = convertcolumnswithdateinname(concurnewestfile)

concurnewestfile.expense_amount__reimbursement_currency_ = (
    concurnewestfile.expense_amount__reimbursement_currency_.astype(float)
)


concurnewestfile = concurnewestfile.assign(
    SpendSource=concurnewestfile.payment_type, DataSource="Concur-Spendcube Entries"
).reset_index(drop=True)

concurallocationnewestfile = columnreplaceforsqlload(concurallocationnewestfile)

concurallocationnewestfile = convertcolumnswithdateinname(concurallocationnewestfile)

# concurallocationnewestfile.expense_amount__reimbursement_currency_ = concurallocationnewestfile.expense_amount__reimbursement_currency_.astype(float)

concurallocationnewestfile = concurallocationnewestfile.reset_index(drop=True)
concurallocationnewestfile = concurallocationnewestfile.loc[
    ~pd.isna(concurallocationnewestfile.percentage)
].drop_duplicates()

concurnewestfile_merged = concurnewestfile.merge(
    concurallocationnewestfile,
    how="left",
    left_on="entry_key",
    right_on="entry_key",
).drop_duplicates()

concurnewestfile_merged_2 = (
    concurnewestfile_merged.assign(
        account_code_1=concurnewestfile_merged.account_code_1.fillna(0).astype(int),
        product_description=concurnewestfile_merged["spend_category"],
        Line_of_Business="",
        report_name=concurnewestfile_merged.report_name.fillna("")
        .astype(str)
        .str.replace("[^A-Za-z0-9 ]+", "", regex=True),
        purpose=concurnewestfile_merged.purpose.fillna("")
        .astype(str)
        .str.replace("[^A-Za-z0-9 ]+", "", regex=True),
        vendor=concurnewestfile_merged.vendor.fillna("")
        .astype(str)
        .str.replace("[^A-Za-z0-9 ]+", "", regex=True),
    )
    .merge(gl, how="left", left_on="account_code_1", right_on="g/l_account")
    .reset_index()
)

In [ ]:
# only one concur file

try:
    concurnewestfile_merged_2 = continue_from_sql_index(
        schemaandtable, concurnewestfile_merged_2, engine
    )
except:
    pass

concurnewestfile_merged_2 = concurnewestfile_merged_2.assign(
    join_key_datasource=lambda concurnewestfile_merged_2: concurnewestfile_merged_2.index.astype(
        str
    )
    + " "
    + concurnewestfile_merged_2.DataSource,
    funding_source=concurnewestfile_merged_2.custom_1___code,
)

concursmall = concurnewestfile_merged_2[
    [
        "vendor",
        "g/l_acct_long_text",
        "product_description",
        "expense_type",
        "Line_of_Business",
        "vendor",
        "entry_key",
        "paid_date_x",
        "expense_amount",
        "DataSource",
        "SpendSource",
        "index",
        "join_key_datasource",
    ]
]

concursmall.columns = column_names

In [ ]:
# only one concur file

concurnewestfile_merged_2.to_sql(
    concur_table, engine, schema=schema, if_exists="append", index=True, chunksize=1000
)

In [ ]:
# all concur and replace
concurnewestfile_merged_2 = concurnewestfile_merged_2.assign(
    join_key_datasource=lambda concurnewestfile_merged_2: concurnewestfile_merged_2.index.astype(
        str
    )
    + " "
    + concurnewestfile_merged_2.DataSource,
    funding_source=concurnewestfile_merged_2.custom_1___code,
)

concursmall = concurnewestfile_merged_2[
    [
        "vendor",
        "g/l_acct_long_text",
        "product_description",
        "expense_type",
        "Line_of_Business",
        "vendor",
        "entry_key",
        "paid_date_x",
        "funding_source",
        "expense_amount",
        "DataSource",
        "SpendSource",
        "index",
        "join_key_datasource",
    ]
]

concursmall.columns = column_names

In [ ]:
# all concur and replace

to_sql(
    concurnewestfile_merged_2,
    "concur",
    creds,
    if_exists="replace",
    bcp_path=r"C:\Program Files\Microsoft SQL Server\Client SDK\ODBC\170\Tools\Binn\bcp.exe",
    work_directory=comballpath,
)

In [ ]:
concurnewestfile_merged_2.to_sql(
    concur_table,
    engine,
    schema=schema,
    if_exists="replace",
    index=True,
    chunksize=1000,
)

In [ ]:
AMEX_Pcard = glob.glob("AMEX-Pcard\*PCard_Amex*.csv")
AMEX_Pcard_csv = LoadFilesFromFolder(AMEX_Pcard, filetype="csv")
AMEX_Pcard_csv = columnreplaceforsqlload(AMEX_Pcard_csv)
AMEX_Pcard_csv = AMEX_Pcard_csv.reset_index(drop=True)
AMEX_Pcard_csv = convertcolumnswithdateinname(AMEX_Pcard_csv)
AMEX_Pcard_csv = (
    (
        AMEX_Pcard_csv.reset_index(drop=True)
        .assign(
            billed_amount=AMEX_Pcard_csv.billed_amount.str.replace(
                "[(),]", "", regex=True
            ).astype(float),
            DataSource="Amex",
            SpendSource="pCard",
            product_cat=AMEX_Pcard_csv.mcc_code.str.replace(
                "\d", "", regex=True
            ).str.replace(",", "", regex=True),
            glaccount=AMEX_Pcard_csv["g/l_account"].fillna(0).astype(int),
            product_desc="",
            line_of_business="",
        )
        .merge(gl, how="left", left_on="glaccount", right_on="g/l_account")
    )
    .reset_index()
    .assign(
        join_key_datasource=lambda AMEX_Pcard_csv_: AMEX_Pcard_csv_.index.astype(str)
        + " "
        + AMEX_Pcard_csv_.DataSource,
        funding_source=AMEX_Pcard_csv.cost_center,
    )
)

amexsmall = AMEX_Pcard_csv[
    [
        "supplier_name",
        "g/l_acct_long_text",
        "product_desc",
        "product_cat",
        "line_of_business",
        "supplier_#",
        "transaction_reference_number",
        "business_process_date",
        "billed_amount",
        "funding_source",
        "DataSource",
        "SpendSource",
        "index",
        "join_key_datasource",
    ]
]

amexsmall.columns = column_names

In [ ]:
to_sql(
    AMEX_Pcard_csv,
    "amex_pcard",
    creds,
    if_exists="replace",
    bcp_path=r"C:\Program Files\Microsoft SQL Server\Client SDK\ODBC\170\Tools\Binn\bcp.exe",
    work_directory=comballpath,
)

In [ ]:
# this is for a monthly load
# if reloading hash out what you don't need

combineddata = pd.concat(
    [
        # concursmall,
        # aribasmall,
        disbursementsmall,
        # boasmall
    ]
)

In [ ]:
# This is for a full reload

combineddata = pd.concat(
    [concursmall, aribasmall, amexsmall, disbursementsmall, srmsmall, boasmall]
)

In [ ]:
# combines to bag of words

combineddata["bag_of_words"] = (
    combineddata[
        [
            "supplier",
            "gl_name",
            "product_description",
            "product_category",
            "line_of_business",
        ]
    ]
    .fillna("")
    .apply(lambda row: " ".join(row.astype(str)), axis=1)
)

In [ ]:
import pandas as pd
import os

# import io
# import boto3
import numpy as np
import glob
import os  # ,time
import glob
import pandas as pd

# import pyodbc
# import datetime
import numpy as np
import getpass
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier


def top10summarizedfunction(dataframe, groupingcolumn: str, numericcolumn: float):
    summarized = dataframe.groupby(groupingcolumn)[numericcolumn].apply(
        lambda x: x.astype(float).sum()
    )
    top10 = summarized[summarized.index != ""].nlargest(20).reset_index()
    return top10


def removespacesfromlist(startinglist):
    newlist = []
    for i in startinglist:
        if i.find(" ") != -1:
            if i.count(" ") > 1:
                listofterms = i.split(" ")
                for j in range(len(listofterms)):
                    newlist.append(listofterms[j])
                    newlist.append(listofterms[0] + "_" + listofterms[j])
            else:
                listofterms = i.split(" ")
                for j in range(len(listofterms)):
                    newlist.append(listofterms[j])
                newlist.append(i.replace(" ", "_"))
        else:
            newlist.append(i)
    return newlist


# parser = argparse.ArgumentParser()
# parser.add_argument('--max_df', type = float, default = .5)
# parser.add_argument('--max_features', type = int, default = None)
# parser.add_argument('--use_idf', type = bool, default = True)
# parser.add_argument('--norm', type = str, default = 'l2')
# parser.add_argument('--alpha', type = float, default = 1e-05)
# parser.add_argument('--penalty', type = str, default = 'elasticnet')
# parser.add_argument('--max_iter', type = int, default = 1000)
# parser.add_argument('--tol', type = float, default = 0.001)
user = getpass.getuser()
os.chdir(
    r"C:\Users"
    + "\\"
    + user
    + "\Boston University\Continuous Improvement & Data Analytics - Documents\Automation and Data"
)
recategorizelist = glob.glob(
    "Spendcube-Training Set\*recategorize*.csv"
)  # * means all if need specific format then *.csv
recategorizedall = pd.DataFrame()
for file in recategorizelist:
    interim = pd.read_csv(file, dtype=str)
    recategorizedall = pd.concat([interim, recategorizedall], axis=0, join="outer")
supplier_name_v = (
    recategorizedall[~pd.isna(recategorizedall["Supplier Name"])].drop(
        [
            "supplier",
            "gl",
            "productdescription",
            "productcategoryname",
            "lineofbusiness",
            "sourcesystem",
            "newcategory",
            "oldcategory",
        ],
        axis=1,
    )
).rename(
    columns={
        "Supplier Name": "supplier",
        "gl_name": "gl_name",
        "product_description   ": "product_description",
        "product_category": "product_category_name",
        "line_of_business": "Line of Business",
        "sourcesystem": "source system",
        "new category": "CategoryL2",
        "CategoryCorrect": "oldcategory",
    }
)

supplier_v = (
    recategorizedall[~pd.isna(recategorizedall["supplier"])]
    .drop(
        [
            "Supplier Name",
            "gl_name",
            "product_description",
            "product_category",
            "line_of_business",
            "source system",
            "new category",
            "CategoryCorrect",
        ],
        axis=1,
    )
    .rename(
        columns={
            "supplier": "supplier",
            "gl": "gl_name",
            "productdescription": "product_description",
            "productcategoryname": "product_category_name",
            "lineofbusiness": "Line of Business",
            "sourcesystem": "source_system",
            "newcategory": "CategoryL2",
        }
    )
)
recat3 = pd.concat([supplier_name_v, supplier_v]).reset_index(drop=True)

recat3["bagofwords"] = (
    recat3["supplier"].str.lower().fillna("")
    + " "
    + recat3["gl_name"].str.lower().fillna("")
    + " "
    + recat3["product_description"].str.lower().fillna("")
    + " "
    + recat3["product_category_name"].str.lower().fillna("")
    + " "
    + recat3["Line of Business"].str.lower().fillna("")
)

train_data = pd.read_csv(
    r"C:\Users\ipoole\Boston University\Continuous Improvement & Data Analytics - Documents\Automation and Data\Spendcube-Training Set\TrainingSetUpdated.csv",
    sep=",",
    encoding="cp1252",
)

train_outer = train_data.merge(
    recat3,
    how="outer",
    left_on="bagofwords",
    right_on="bagofwords",
    suffixes=("", "_drop"),
    indicator=True,
)
train_outer_left = train_outer[train_outer._merge == "left_only"]
train2 = train_outer_left.loc[:, ~train_outer_left.columns.str.endswith("_drop")]
recat3 = recat3[train_data.columns.to_list()]
recategorized_all = pd.concat([train2, recat3])
train_data = pd.read_csv(
    r"C:\Users\ipoole\Boston University\Continuous Improvement & Data Analytics - Documents\Automation and Data\Spendcube-Training Set\TrainingSetUpdated.csv",
    sep=",",
    encoding="cp1252",
)
Folder = glob.glob(
    "Spendcube-Training Set/*recategorize*.csv"
)  # * means all if need specific format then *.csv
for to_recategorize in Folder:
    recat = pd.read_csv(to_recategorize, dtype=str)
    recat = recat.drop(columns=["oldcategory"])
    recatcolumnnames = [
        "gl_name",
        "Line of Business",
        "product_category_name",
        "source_system",
        "supplier",
        "product_description",
        "CategoryL2",
    ]
    recat.columns = recatcolumnnames
    recat["bagofwords"] = (
        recat["supplier"].str.lower()
        + " "
        + recat["gl_name"].str.lower()
        + " "
        + recat["product_description"].str.lower()
        + " "
        + recat["product_category_name"].str.lower()
        + " "
        + recat["Line of Business"].str.lower()
    )
    train_data = pd.concat([train_data, recat])
train_y = recategorized_all["CategoryL2"]
train_X = recategorized_all["bagofwords"]

clf = Pipeline(
    [
        (
            "vect",
            CountVectorizer(
                encoding="utf-8",
                decode_error="ignore",
                max_df=0.5,
                max_features=None,
                ngram_range=(1, 1),
                stop_words="english",
            ),
        ),
        ("tfidf", TfidfTransformer(use_idf=True, norm="l2")),
        (
            "clf",
            SGDClassifier(
                loss="hinge",
                class_weight="balanced",
                alpha=1e-05,
                penalty="elasticnet",
                max_iter=1000,
                tol=0.001,
            ),
        ),
    ]
)

clf = clf.fit(train_X.astype("U"), train_y.values.astype("U"))

In [ ]:
# Testing Version
from joblib import dump

dump(clf, "Python-AutomationEXEs/Spendcube_Algorithm/SpendCube_ML_testV.joblib")

In [ ]:
# Prod Version commented to protect from unwanted changes
from joblib import dump

# dump(clf, 'Python-AutomationEXEs/Spendcube_Algorithm/SpendCube_ML.joblib')

In [ ]:
# Load both
from joblib import load

clf = load("Python-AutomationEXEs/Spendcube_Algorithm/SpendCube_ML.joblib")
# test_clf = load('Python-AutomationEXEs/Spendcube_Algorithm/SpendCube_ML_testV.joblib')

In [ ]:
def predict_data(algo, start_df, column):
    predicted_svm = algo.predict(start_df[column])

    # turns the predictions to a dataframe, renames the columns and adds an index
    predicted_df = pd.DataFrame(predicted_svm)
    predicted_df.rename(columns={"0": "category"}, inplace=True)

    predictions = pd.concat([combineddata.reset_index(drop=True), predicted_df], axis=1)

    return predictions

In [ ]:
predict_table = "predictions"

predictions = predict_data(clf, combineddata, "bag_of_words")

In [ ]:
test_predicitons = predict_data(test_clf, combineddata, "bag_of_words")

In [ ]:
predictions.to_csv(
    r"C:\Users\ipoole\Boston University\Continuous Improvement & Data Analytics - Documents\Automation and Data\CombAll\predictions_python_3.csv",
    sep="|",
    doublequote=True,
)

In [ ]:
predictions = predictions.replace({",": ""}, regex=True)
predictions.columns

In [ ]:
try:
    predictions = continue_from_sql_index(schemaandtable, predictions, engine)
except:
    pass

predictions = predictions.replace({",": ""}, regex=True)

In [ ]:
schema

In [ ]:
predictions.to_sql(
    predict_table, engine, schema=schema, if_exists="append", index=True, chunksize=1000
)

In [ ]:
predictions2 = predictions.replace({",": ""}, regex=True)

to_sql(
    predictions2,
    "predictions",
    creds,
    if_exists="replace",
    bcp_path=r"C:\Program Files\Microsoft SQL Server\Client SDK\ODBC\170\Tools\Binn\bcp.exe",
    work_directory=comballpath,
)

In [ ]:
predictions2 = predictions.replace({",": ""}, regex=True)

predictions2.to_sql(
    table,
    engine,
    schema,
    if_exists="replace",
    index=True,
    chunksize=2000,
    method="multi",
)

In [ ]:
import unicodedata
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import nltk

# List of keywords to help identify stop_words
vendor_stopwords = [
    "biz",
    "bv",
    "co",
    "comp",
    "company",
    "corp",
    "corporation",
    "dba",
    "inc",
    "incorp",
    "incorporat",
    "incorporate",
    "incorporated",
    "incorporation",
    "international",
    "intl",
    "intnl",
    "limited",
    "llc",
    "ltd",
    "llp",
    "machines",
    "pvt",
    "pte",
    "private",
    "unknown",
]


# Text data encoder function
def filter_ascii(text):
    return (
        unicodedata.normalize("NFKD", text)
        .encode("ascii", "ignore")
        .decode("utf-8", "ignore")
    )


# Remove spl characters & digits (optional) function
def remove_special_characters(text, remove_digits=False):
    pattern = r"[^a-zA-z0-9\s]" if not remove_digits else r"[^a-zA-z\s]"
    text = re.sub(pattern, "", text)
    return text


# Remove vendor specific stop words
def clean_stopwords(text, eng=False):
    if eng == False:
        custom = vendor_stopwords
    else:
        custom = vendor_stopwords + list(ENGLISH_STOP_WORDS)
    for x in custom:
        pattern2 = r"\b" + x + r"\b"
        text = re.sub(pattern2, "", text)
    return text


# Trim the text to remove spaces
def clean_spaces(text):
    text = text.replace("  ", " ")
    text = text.strip()
    return text


# Function to Preprocess Textual data. Provide input as df['Column Name'] to this function
def preprocess_text(column, remove_digits=True, lemm=True, eng=False):
    try:
        column = [filter_ascii(text) for text in column]
        column = [remove_special_characters(text, remove_digits) for text in column]
        column = [text.lower() for text in column]
        column = [clean_stopwords(text, eng) for text in column]
        column = [clean_spaces(text) for text in column]
        ## Lemmatisation (convert the word into root word)
        if lemm == True:
            lem = nltk.stem.wordnet.WordNetLemmatizer()
            column = [lem.lemmatize(text) for text in column]
        return column
    except Exception as e:
        return print(e)

In [ ]:
table = "predictions"
schema = "dbo"

cur_predictions = pd.read_sql("SELECT [index], supplier FROM " + table, engine)

In [ ]:
cur_predictions = cur_predictions.fillna("")

In [ ]:
cur_predictions["normsupplier"] = preprocess_text(cur_predictions.supplier)

In [ ]:
from fuzzywuzzy import fuzz


# Function to generate similarity matrix. Provide input as df['Column Name'] to this function
def fuzz_similarity(column):
    similarity_array = np.ones((len(column), (len(column)))) * 100
    for i in range(1, len(column)):
        for j in range(i):
            s1 = fuzz.token_set_ratio(column[i], column[j]) + 0.00000000001
            s2 = fuzz.partial_ratio(column[i], column[j]) + 0.00000000001
            similarity_array[i][j] = 2 * s1 * s2 / (s1 + s2)

    for i in range(len(column)):
        for j in range(i + 1, len(column)):
            similarity_array[i][j] = similarity_array[j][i]
            np.fill_diagonal(similarity_array, 100)
    return similarity_array

In [ ]:
cur_predictions = cur_predictions.drop_duplicates(subset=["normsupplier"])

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import AffinityPropagation

vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(cur_predictions["normsupplier"])

In [ ]:
affin = AffinityPropagation()
affin.fit_predict(X)

In [ ]:
cur_predictions = cur_predictions[
    cur_predictions["normsupplier"].apply(lambda x: len(x) > 7)
]

In [ ]:
cur_predictions.to_csv("curpredsuppliernorm.csv")